# 03. Huấn luyện Mô hình & Đánh giá dự phòng

Notebook này thực hiện quá trình huấn luyện và xác thực các lớp mô hình dự báo từ cơ bản đến nâng cao:

1. **Nhóm mô hình cơ sở (Baselines)**:
   - Naive (Dự báo bằng giá trị cuối cùng $y_t$)
   - Moving Average (Trung bình trượt 24 giờ tương ứng cửa sổ 96 bước)
   - Hồi quy tuyến tính Baseline (chỉ sử dụng các biến trễ)

2. **Nhóm mô hình máy học chứa yếu tố mùa vụ (Seasonality / ML / SARIMAX)**:
   - Hồi quy tuyến tính kết hợp biến mùa vụ Fourier
   - XGBoost Regressor (Huấn luyện trên tất cả các đặc trưng lịch, Fourier và trễ)
   - SARIMAX(1,1,1)(1,0,0,24) (Huấn luyện trên dữ liệu downsampled theo giờ để tối ưu tốc độ)

3. **Mô hình học sâu nâng cao (Advanced Deep Learning)**:
   - PyTorch LSTM (sử dụng chuỗi lịch sử lookback window dài 96 bước)
   - PyTorch GRU (sử dụng chuỗi lịch sử lookback window dài 96 bước)
   - PyTorch Transformer (sử dụng chuỗi lịch sử lookback window dài 96 bước)

In [1]:
import sys
sys.path.append('../')

import os
import numpy as np
import pandas as pd
import xgboost as xgb
import torch
import matplotlib.pyplot as plt

from src.models import NaiveModel, MovingAverageModel, train_linear_regression, PyTorchSeqRegressor

## 1. Tải các tập dữ liệu đã phân tách

In [2]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

print(f"Kích thước Train: {train_df.shape}")
print(f"Kích thước Val: {val_df.shape}")
print(f"Kích thước Test: {test_df.shape}")

Kích thước Train: (48305, 26)
Kích thước Val: (10351, 26)
Kích thước Test: (10352, 26)


## 2. Phân loại đặc trưng và Biến mục tiêu

In [3]:
target_col = 'OT'
raw_features = ['HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL']
calendar_features = ['hour', 'dayofweek', 'month', 'is_weekend']
fourier_features = ['sin_24', 'cos_24', 'sin_168', 'cos_168']
lag_features = [col for col in train_df.columns if 'lag' in col]
rolling_features = [col for col in train_df.columns if 'roll' in col]

y_train = train_df[target_col]
y_val = val_df[target_col]
y_test = test_df[target_col]

## 3. Huấn luyện các mô hình cơ sở (Baselines)

In [4]:
naive_model = NaiveModel()
naive_preds = naive_model.predict(test_df)

ma_model = MovingAverageModel(window=96)
ma_preds = ma_model.predict(test_df)

X_train_lr_base = train_df[lag_features]
X_test_lr_base = test_df[lag_features]
lr_base_model = train_linear_regression(X_train_lr_base, y_train)
lr_base_preds = lr_base_model.predict(X_test_lr_base)

## 4. Huấn luyện mô hình Mùa vụ / Máy học (ML)

In [5]:
fourier_lr_features = lag_features + fourier_features
X_train_lr_fourier = train_df[fourier_lr_features]
X_test_lr_fourier = test_df[fourier_lr_features]
lr_fourier_model = train_linear_regression(X_train_lr_fourier, y_train)
lr_fourier_preds = lr_fourier_model.predict(X_test_lr_fourier)

all_features = raw_features + calendar_features + fourier_features + lag_features + rolling_features
X_train_xgb = train_df[all_features]
X_test_xgb = test_df[all_features]

xgb_model = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train_xgb, y_train)
xgb_preds = xgb_model.predict(X_test_xgb)

# Vẽ biểu đồ feature importance của XGBoost
importances = xgb_model.feature_importances_
feature_names = X_train_xgb.columns
imp_df = pd.DataFrame({"feature_name": feature_names, "importance": importances})
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

plt.figure(figsize=(10, 6))
plt.barh(imp_df["feature_name"].head(20), imp_df["importance"].head(20), color="steelblue")
plt.gca().invert_yaxis()
plt.title("XGBoost Feature Importance - Top 20 (Gain)")
plt.xlabel("Tầm quan trọng (Importance)")
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/xgb_feature_importance.png", dpi=120)
plt.close()
print("Đã lưu biểu đồ feature importance vào figures/xgb_feature_importance.png")

# --- Huấn luyện SARIMAX (giảm mẫu theo giờ để tránh chạy quá lâu) ---
try:
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    print("Bắt đầu huấn luyện SARIMAX(1,1,1)(1,0,0,24)... (Downsampled hourly)")
    STEP, N_TR, N_TE = 4, 720, 240  # 15-phút -> giờ, 30 ngày train, 10 ngày test
    train_h = train_df.iloc[::STEP].reset_index(drop=True)
    test_h  = test_df.iloc[::STEP].reset_index(drop=True)

    def _fourier_hourly(dt_series, n=2):
        dt_series = pd.to_datetime(dt_series)
        fod = (dt_series.dt.hour + dt_series.dt.minute / 60) / 24.0
        cols = []
        for k in range(1, n + 1):
            cols.append(np.sin(2 * np.pi * k * fod.values))
            cols.append(np.cos(2 * np.pi * k * fod.values))
        return np.column_stack(cols)

    exog_tr = _fourier_hourly(train_h["date"].iloc[-N_TR:])
    exog_te = _fourier_hourly(test_h["date"].iloc[:N_TE])

    sarimax_model = SARIMAX(
        train_h[target_col].values[-N_TR:],
        exog=exog_tr,
        order=(1, 1, 1), seasonal_order=(1, 0, 0, 24),
        enforce_stationarity=False, enforce_invertibility=False
    ).fit(disp=False)

    sarimax_preds = sarimax_model.forecast(steps=N_TE, exog=exog_te)
    sarimax_true = test_h[target_col].values[:N_TE]

    sarimax_pred_df = pd.DataFrame({
        "date": test_h["date"].iloc[:N_TE],
        "y_true": sarimax_true,
        "y_pred": sarimax_preds
    })
    os.makedirs('../results', exist_ok=True)
    sarimax_pred_df.to_csv('../results/predictions_sarimax.csv', index=False)
    print("✓ Huấn luyện và lưu kết quả SARIMAX thành công!")
except Exception as e:
    print(f"Lỗi khi chạy SARIMAX: {e}")


Đã lưu biểu đồ feature importance vào figures/xgb_feature_importance.png


Bắt đầu huấn luyện SARIMAX(1,1,1)(1,0,0,24)... (Downsampled hourly)


✓ Huấn luyện và lưu kết quả SARIMAX thành công!


## 5. Huấn luyện các mô hình học sâu nâng cao (PyTorch LSTM, GRU, Transformer với L=96)

In [6]:
def create_sequences(df, target_col, feature_cols, seq_len=96):
    X, y = [], []
    feature_data = df[feature_cols].values
    target_data = df[target_col].values
    for i in range(len(df) - seq_len):
        X.append(feature_data[i : i + seq_len])
        y.append(target_data[i + seq_len])
    return np.array(X), np.array(y)

lstm_features = raw_features + calendar_features + fourier_features + rolling_features
seq_len = 96

X_train_seq, y_train_seq = create_sequences(train_df, target_col, lstm_features, seq_len)
X_val_seq, y_val_seq = create_sequences(val_df, target_col, lstm_features, seq_len)
X_test_seq, y_test_seq = create_sequences(test_df, target_col, lstm_features, seq_len)

print("Kích thước Train X:", X_train_seq.shape, "y:", y_train_seq.shape)
print("Kích thước Val X:", X_val_seq.shape, "y:", y_val_seq.shape)
print("Kích thước Test X:", X_test_seq.shape, "y:", y_test_seq.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
import matplotlib.pyplot as plt
import os
os.makedirs('../figures', exist_ok=True)

# 1. Huấn luyện LSTM
lstm_regressor = PyTorchSeqRegressor(
    model_type="lstm",
    input_dim=len(lstm_features),
    hidden_dim=128,
    num_layers=2,
    lr=0.001,
    epochs=50,
    patience=10,
    batch_size=64,
    device=device,
    checkpoint_dir="../checkpoints"
)
tr_loss, val_loss = lstm_regressor.fit(X_train_seq, y_train_seq, X_val_seq, y_val_seq)
lstm_preds = lstm_regressor.predict(X_test_seq)

# Vẽ đường cong học tập LSTM
plt.figure(figsize=(8, 4))
plt.plot(tr_loss, label="Train")
plt.plot(val_loss, label="Val")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("LSTM - Learning Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/learning_curve_LSTM.png", dpi=120)
plt.close()
print("Đường cong học tập LSTM đã lưu vào figures/learning_curve_LSTM.png")

# 2. Huấn luyện GRU
gru_regressor = PyTorchSeqRegressor(
    model_type="gru",
    input_dim=len(lstm_features),
    hidden_dim=128,
    num_layers=2,
    lr=0.001,
    epochs=50,
    patience=10,
    batch_size=64,
    device=device,
    checkpoint_dir="../checkpoints"
)
tr_loss, val_loss = gru_regressor.fit(X_train_seq, y_train_seq, X_val_seq, y_val_seq)
gru_preds = gru_regressor.predict(X_test_seq)

# Vẽ đường cong học tập GRU
plt.figure(figsize=(8, 4))
plt.plot(tr_loss, label="Train")
plt.plot(val_loss, label="Val")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("GRU - Learning Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/learning_curve_GRU.png", dpi=120)
plt.close()
print("Đường cong học tập GRU đã lưu vào figures/learning_curve_GRU.png")

# 3. Huấn luyện Transformer
transformer_regressor = PyTorchSeqRegressor(
    model_type="transformer",
    input_dim=len(lstm_features),
    d_model=64,
    nhead=4,
    num_layers=2,
    lr=0.001,
    epochs=50,
    patience=10,
    batch_size=64,
    device=device,
    checkpoint_dir="../checkpoints"
)
tr_loss, val_loss = transformer_regressor.fit(X_train_seq, y_train_seq, X_val_seq, y_val_seq)
transformer_preds = transformer_regressor.predict(X_test_seq)

# Vẽ đường cong học tập Transformer
plt.figure(figsize=(8, 4))
plt.plot(tr_loss, label="Train")
plt.plot(val_loss, label="Val")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("Transformer - Learning Curve")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/learning_curve_Transformer.png", dpi=120)
plt.close()
print("Đường cong học tập Transformer đã lưu vào figures/learning_curve_Transformer.png")

aligned_test_df = test_df.iloc[seq_len:].reset_index(drop=True)
y_test_aligned = y_test_seq

Kích thước Train X: (48209, 96, 18) y: (48209,)
Kích thước Val X: (10255, 96, 18) y: (10255,)
Kích thước Test X: (10256, 96, 18) y: (10256,)


Huấn luyện LSTM (max 50 epoch, patience=10)


  Epoch   1/ 50: train=0.063203 | val=0.104253


  Epoch   5/ 50: train=0.011139 | val=0.085849


  Epoch  10/ 50: train=0.005787 | val=0.099293


...

  Epoch   1/ 50: train=0.059260 | val=0.074248


  Epoch   5/ 50: train=0.013040 | val=0.078071


  Epoch  10/ 50: train=0.008315 | val=0.086229


  Early stopping tại epoch 12 (best val=0.071146)


Đường cong học tập Transformer đã lưu vào figures/learning_curve_Transformer.png


## 6. Lưu trữ kết quả dự báo của tất cả mô hình

In [7]:
predictions_df = pd.DataFrame({
    "date": aligned_test_df["date"],
    "y_true": y_test_aligned,
    "naive": naive_preds[seq_len:],
    "moving_average": ma_preds[seq_len:],
    "linear_regression_baseline": lr_base_preds[seq_len:],
    "linear_regression_fourier": lr_fourier_preds[seq_len:],
    "xgboost": xgb_preds[seq_len:],
    "lstm": lstm_preds,
    "gru": gru_preds,
    "transformer": transformer_preds
})

os.makedirs('../results', exist_ok=True)
predictions_df.to_csv('../results/predictions.csv', index=False)
print("Lưu tệp predictions.csv thành công.")

Lưu tệp predictions.csv thành công.
